# Interfacial Energy

Calculate the interfacial energy γ (eV/Å²) of a heterostructure using a multi-material DFT workflow on the Mat3ra platform.

The workflow takes **three materials, in order**:

- **[0] Interface** — the heterostructure (e.g. Gr/Ni(111)).
- **[1] Substrate bulk** — the substrate reference crystal.
- **[2] Film bulk** — the film reference crystal.

Only the interface is computed here (`pw_scf`). The two bulk reference energies are fetched from previously-finished **Total Energy** jobs, so each bulk must already be converged/relaxed and have such a job on the platform. The substrate/film multiplicities are derived automatically from the structures; `N_INTERFACES` (1 with vacuum, 2 for a periodic stack) is provided by this notebook.

This notebook loads an interface from `../uploads`, resolves the substrate and film bulk crystals from the interface `metadata.build`, verifies each bulk has a total energy on the platform, and submits the three materials to the multi-material workflow.

<h2 style="color:green">Usage</h2>

1. Build an interface (for example via `create_interface_with_min_strain_zsl.ipynb`) and export it to `../uploads`, or set `INTERFACE_NAME` to match an interface on the platform.
2. Run [Total Energy](total_energy.ipynb) for each resolved substrate and film bulk crystal.
3. Set parameters in cells 1.2 and 1.3 below.
4. Click "Run" > "Run All".
5. Inspect the resolved bulks, their total energies, and the final interfacial energy.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load the interface, resolve substrate/film bulks with their total energies, and assemble the three job materials.
4. Configure the Interfacial Energy workflow.
5. Configure compute.
6. Create, submit, and monitor the multi-material job.
7. Retrieve results.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"  # Name to search in uploads or on the platform

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "interfacial_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Interfacial Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


### 1.3. Set specific interfacial energy parameters

In [ ]:
# K-grid for interface SCF (if not set, KPPRA is used by default)
SCF_KGRID = None  # e.g., [4, 4, 1]

# Number of interfaces in the supercell: 1 if the slab has vacuum (one physical
# interface), 2 for a periodic stack with no vacuum. Substrate and film
# multiplicities are derived automatically from the structures.
N_INTERFACES = 1

## 2. Authenticate and initialize API client
### 2.1. Authenticate


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account to work under


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load interface and resolve substrate/film bulks
### 3.1. Load interface from local file or platform


In [ ]:
import re
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

interface = load_material_from_folder(FOLDER, INTERFACE_NAME)

if interface is None:
    interface_matches = client.materials.list({
        "name": {"$regex": re.escape(INTERFACE_NAME), "$options": "i"},
    })
    if not interface_matches:
        raise ValueError(
            f"No interface containing '{INTERFACE_NAME}' was found in '{FOLDER}' or on the platform."
        )
    interface = Material.create(interface_matches[0])
    print(f"♻️  Loaded interface from platform: {interface_matches[0]['_id']}")
else:
    print(f"✅ Loaded interface from folder: {interface.name}")

visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")


### 3.2. Prepare interface for Quantum ESPRESSO
Strip atom labels. Labels are useful for analysis notebooks but are not compatible with QE input generation.


In [ ]:
interface_material = interface.clone()
interface_material.basis.set_labels_from_list([])
print(f"Prepared interface material: {interface_material.name}")
visualize(interface_material, repetitions=[1, 1, 1], rotation="-90x")


### 3.3. Load workflow from Standata


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

interfacial_workflow_config = WorkflowStandata.filter_by_application(APPLICATION_NAME).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)
interfacial_workflow = Workflow.create(interfacial_workflow_config)
print(f"Loaded workflow: {interfacial_workflow.name}")
print(f"Multi-material: {getattr(interfacial_workflow, 'isMultiMaterial', False)}")

### 3.4. Resolve substrate/film crystals from interface build metadata
Uses `made`'s `get_interface_bulk_crystal` helper for typed access to the interface build configuration, instead of hand-parsing the raw `metadata.build` dict.

In [ ]:
from mat3ra.made.tools.analyze.interface_material import get_interface_bulk_crystal
from mat3ra.made.tools.build_components.metadata import MaterialWithBuildMetadata

interface_with_metadata = MaterialWithBuildMetadata.create(interface_material.to_dict())

# Bulk crystals as recorded when the interface was built.
CRYSTAL_SUBSTRATE = Material.create(get_interface_bulk_crystal(interface_with_metadata, part="substrate"))
CRYSTAL_FILM = Material.create(get_interface_bulk_crystal(interface_with_metadata, part="film"))

print(f"Substrate crystal: {CRYSTAL_SUBSTRATE.name}")
print(f"Film crystal: {CRYSTAL_FILM.name}")

### 3.5. Find substrate and film bulk materials on the platform
Each bulk must already exist on the platform with a finished Total Energy job (see [Total Energy](total_energy.ipynb)); the workflow will fetch that energy rather than recomputing it.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.analysis import find_bulk_with_total_energy

substrate_bulk, substrate_te_property = find_bulk_with_total_energy(
    client, ACCOUNT_ID, bulk_crystal=CRYSTAL_SUBSTRATE, role="substrate", material_index=1
)
film_bulk, film_te_property = find_bulk_with_total_energy(
    client, ACCOUNT_ID, bulk_crystal=CRYSTAL_FILM, role="film", material_index=2
)

visualize([substrate_bulk, film_bulk], repetitions=[1, 1, 1])

### 3.6. Save the interface and assemble the job materials
Save the (label-stripped) interface, then assemble the three materials the multi-material workflow expects, in order: `[0]` interface, `[1]` substrate bulk, `[2]` film bulk.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_interface_response = get_or_create_material(client, interface_material, ACCOUNT_ID)
saved_interface = Material.create(saved_interface_response)

# Order matters: the workflow reads material 0 as the interface, 1 as the
# substrate bulk, and 2 as the film bulk.
materials = [saved_interface, substrate_bulk, film_bulk]
print("✅ Materials for the multi-material job (index → material):")
for index, material in enumerate(materials):
    print(f"  [{index}] {material.name} ({material.id})")

visualize(materials, repetitions=[1, 1, 1], rotation="-90x")

## 4. Configure the Interfacial Energy workflow
### 4.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 4.2. Configure workflow and preview it

In [ ]:
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

if type(N_INTERFACES) is not int or N_INTERFACES not in (1, 2):
    raise ValueError("N_INTERFACES must be either 1 (vacuum) or 2 (periodic stack)")

interfacial_workflow.name = MY_WORKFLOW_NAME
interfacial_workflow = apply_scf_kgrid(interfacial_workflow, scf_kgrid=SCF_KGRID)

for subworkflow in interfacial_workflow.subworkflows:
    if "set-n-interfaces" not in [unit.name for unit in subworkflow.units]:
        continue
    unit = subworkflow.get_unit_by_name(name="set-n-interfaces")
    unit.value = str(N_INTERFACES)
    subworkflow.set_unit(unit)

visualize_workflow(interfacial_workflow)

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 6. Create the Interfacial Energy job
### 6.1. Create the multi-material job (interface + substrate bulk + film bulk)

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON
from mat3ra.utils.namespace import dict_to_namespace_recursive

print("Materials (job material order):")
for index, material in enumerate(materials):
    print(f"  [{index}] {material.name} ({material.id})")
print(f"Project: {project_id}")

interfacial_job_name = f"{MY_WORKFLOW_NAME} {saved_interface.formula} {timestamp}"
interfacial_job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=interfacial_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=interfacial_job_name,
    compute=compute.to_dict(),
)

interfacial_job = dict_to_namespace_recursive(interfacial_job_response)
interfacial_job_id = interfacial_job._id
print(f"✅ Interfacial Energy job created successfully: {interfacial_job_id}")
display_JSON(interfacial_job_response)

### 6.2. Submit the Interfacial Energy job and monitor the status

In [ ]:
client.jobs.submit(interfacial_job_id)
print(f"✅ Job {interfacial_job_id} submitted successfully!")


In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [interfacial_job_id], poll_interval=POLL_INTERVAL)


## 7. Retrieve results
### 7.1. Retrieve and visualize interfacial energy

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

interfacial_energy_data = client.properties.get_for_job(
    interfacial_job_id, property_name="interfacial_energy"
)
interface_te_data = client.properties.get_for_job(
    interfacial_job_id, property_name=PropertyName.scalar.total_energy.value
)

visualize_properties(interfacial_energy_data, title="Interfacial Energy")

print(f"Interface (material 0):     {saved_interface.name} ({saved_interface.id})")
print(f"Substrate bulk (material 1): {substrate_bulk.name} ({substrate_bulk.id}), "
      f"total energy: {substrate_te_property['data']['value']} eV")
print(f"Film bulk (material 2):      {film_bulk.name} ({film_bulk.id}), "
      f"total energy: {film_te_property['data']['value']} eV")